In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import datetime
import pandas as pd
import matplotlib as mpl

sys.path.insert(0,"../src/")
import common.d2d as d2d

# from run_selection_single_channel import RunInfo
# import WaveformProcessor
# import FitSPE

# from common.utils import vec_regex_search

%run /home/ws/sk6801/sw/UCSD_analysis/SandyAQ/python_wrappers/notebooks/plot_style_kalinka.py



In [ ]:
def create_color_scheme(color_map: str, array: object, color_range=(0,1), darken=1, reverse=False):
    values = sorted(np.unique(array))
    if reverse:
        values = values[::-1]
        
    cmap = plt.get_cmap(color_map)
    color = cmap(np.linspace(color_range[0], color_range[1], len(values)))
    
    # https://stackoverflow.com/questions/37517587/how-can-i-change-the-intensity-of-a-colormap-in-matplotlib
    color[:,0:3] *= darken
    
    clr = {values[i]: color[i] for i in range(len(values))}
    return clr

In [ ]:
# before
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_dataplotsgain_info_single_channel_20240920.csv"

# after changing the SPE threshold
path_GXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250608_all_gain_info_single_channel.csv"
# path_GXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250812_all_gain_info_single_channel.csv"
# path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250609_LXe_gain_info_single_channel.csv"
# path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250610_LXe_gain_info_single_channel.csv"
path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250616_LXe_gain_info_single_channel.csv"
# path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250706_LXe_gain_info_single_channel.csv"
# path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250812_LXe_gain_info_single_channel.csv"

df_GXe = pd.read_csv(path_GXe, 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")

df_LXe = pd.read_csv(path_LXe, 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")


df_all = pd.concat([df_GXe, df_LXe], axis=0)   


In [ ]:
class GainAnalysis:
    def __init__(self, df, output_path=None):
        self.df = df
        self.info = d2d.data(df)
        self.output_path = output_path
        
        self.settings()
        self.data_selection()
        self.voltage_calibration(calibration = False)
        self.create_dataframe()
        self.calculate_breakdown_voltage(output_path=self.output_path)


    def settings(self):
        self.color_temperature = create_color_scheme(
            "coolwarm_r", 
            self.info.temperature_K,
            darken = 0.8,
            color_range=(0.2,0.9),
            reverse=True)
        
        self.color_channel = create_color_scheme("viridis", 
                                    self.info.channel)

        self.color_voltage = create_color_scheme("hot", 
                                    self.info.voltage_preamp1_V,
                                    color_range=(0,0.8))

        self.dict_preamp_channel = {1: [0,1,2,3,4],
                       2: [5,6,7,8,10],
                       3: [9,11,12,13,14],
                       4: [15,16,17,18,19],
                       5: [20,21,22,23]}
        
        self.date_power_supply_changed = np.datetime64('2024-08-13')
        
    def data_selection(self):
        mask = (self.info.voltage_preamp1_V < -46) & (self.info.baseline_std_V < 0.025) & ~np.isnan(self.info.gain)
        self.info = self.info.apply_mask(mask)
        print("1: ", len(self.info))

        # remove files
        path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_new/20241030_all_1_T98_all_voltages_6.0sig/meta_config_all_20241030_170524.json"
        mask = ~(self.info.md_full_path == path)
        self.info = self.info.apply_mask(mask)

        paths = ['/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_new/20241030_all_1_T98_all_voltages_6.0sig/meta_config_all_20241030_171835.json',
            '/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_tritium/20241031_all_1_T98_all_voltages_6.0sig/meta_config_all_20241031_113258.json']

        for path in paths:
            mask = ~((self.info.md_full_path==path) & (self.info.channel == 2))
            self.info = self.info.apply_mask(mask)

    def voltage_calibration(self, calibration: bool):
        if calibration == True:

            preamp_boards = np.arange(1,6)

            info_corrected = self.info.copy()
            info_corrected.voltage_preamp1_V = info_corrected.voltage_preamp1_V.astype(dtype=float)

            mask_time = (info_corrected.date_time > self.date_power_supply_changed)

            for i, set_voltage in enumerate(df_bias_V_correction_avg["set"]):
                select_voltage = info_corrected.voltage_preamp1_V == set_voltage
                
                for preamp_board in preamp_boards:
                    
                    in_board_mask = np.zeros(len(info_corrected), dtype=bool)
                    for channel in self.dict_preamp_channel[preamp_board]:
                        in_board = info_corrected.channel == channel
                        in_board_mask = in_board_mask | in_board
                    
                    mask = select_voltage & in_board_mask & mask_time
                    
                    corrected_voltage = df_bias_V_correction_avg.iloc[i][f"meas{preamp_board}"]
                    
                    # modify the voltage according to preamp board and set voltage
                    info_corrected.voltage_preamp1_V[mask] = corrected_voltage

            # # also get the df of into_corrected
            # info_corrected_df = info_corrected.get_df()
            # info_corrected_df["breakdown_voltage_V"] = pd.Series(dtype='float')
            # info_corrected_df["over_voltage_V"] = pd.Series(dtype='float')

        else: 
            self.info_corrected = self.info.copy()

    def create_dataframe(self):

        self.info_corrected_df = self.info_corrected.get_df().copy()
        
        # sort again and reset index
        self.info_corrected_df.sort_values("date_time", inplace=True)
        self.info_corrected_df.reset_index(drop=True, inplace=True)
        self.info_corrected_df["run_id"] = self.info_corrected_df.index + int(1)

        # setup columns for results
        self.info_corrected_df["breakdown_voltage_V"] = pd.Series(dtype='float')
        self.info_corrected_df["over_voltage_V"] = pd.Series(dtype='float')
        self.info_corrected_df["junction_capacity"] = pd.Series(dtype='float')
        self.info_corrected_df["total_capacity"] = pd.Series(dtype='float')
        self.info_corrected_df["date_str"] = pd.Series(dtype='float')
        
        # cluster data by date
        self.info_corrected_df["date_str"] = self.info_corrected_df["date_time_str"].str[:7]
        # print(np.unique(self.info_corrected_df.date_str))

        # overwrite the info_corrected with the new df
        self.info_corrected = d2d.data(self.info_corrected_df)

    def calculate_breakdown_voltage(self, output_path):

        if self.info_corrected is None:
            raise ValueError("info_corrected is not set. Please run the data selection and voltage calibration first.")
    
        # select data before and after change of power supply
        mask = self.info_corrected.date_time < self.date_power_supply_changed
        masked_before = self.info_corrected.apply_mask(mask)

        mask = self.info_corrected.date_time > self.date_power_supply_changed
        masked_after = self.info_corrected.apply_mask(mask)

        selected_data = [masked_before, masked_after]

        # temperatures
        temperature = np.unique(self.info_corrected.temperature_K)

        # loop through data sets and calculate breakdown voltage based on the date_cluster
        for data_set in selected_data:
            for tempe in temperature:
                mask = data_set.temperature_K == tempe
                masked_temp = data_set.apply_mask(mask)

                for i, channel in enumerate(np.unique(masked_temp.channel)):
                    mask = masked_temp.channel == channel
                    masked_channel = masked_temp.apply_mask(mask)
                    
                    for date_dataset in np.sort(np.unique(masked_channel.date_str)):
                        mask = masked_channel.date_str == date_dataset
                        masked_date = masked_channel.apply_mask(mask)

                        voltage_list = np.unique(masked_date.voltage_preamp1_V)
                        # n_points.append(len(voltage_list))
                        # date_str_list.append(date_dataset)
                        
                        if len(voltage_list) >= 3:
                            # print("Voltage points: ", voltage_list)

                            # fit
                            if masked_date.voltage_preamp1_V[0] < 0:
                                coef, res, _, _, _ = np.polyfit(-masked_date.voltage_preamp1_V,masked_date.gain,1, full=True)
                            else:
                                coef, res, _, _, _ = np.polyfit(masked_date.voltage_preamp1_V,masked_date.gain,1, full=True)

                            # print("Date: ", date_dataset, " Channel: ", channel, " Temperature: ", tempe, " Coefficients: ", coef)
                            # print("Length of filtered data: " , len(masked_date))

                            breakdown_voltage = -coef[1]/coef[0]
                            # print("Breakdown voltage: ", breakdown_voltage, " for date: ", date_dataset, " channel: ", channel, " temperature: ", tempe)


                            if breakdown_voltage > 0:
                                run_id_list = masked_date.run_id
                                # length += len(run_id_list)
                                

                                for id in run_id_list:
                                    path = self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "md_full_path"].values
                                    assert path in masked_date.md_full_path
                                    abs_bias_voltage = abs(self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "voltage_preamp1_V"])
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "breakdown_voltage_V"] = breakdown_voltage
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "over_voltage_V"] = abs_bias_voltage - breakdown_voltage
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "junction_capacity"] = -coef[0]*1.6e-19/33  # in F
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "total_capacity"] = -coef[0]*1.6e-19  # in F
                                    # self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "coef0"] = coef[0]
                                    # self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "coef1"] = coef[1]

                                
                            
                            
                            # print("Number of runs: ", len(run_id_list))
                            # print("Breakdown voltage: ", breakdown_voltage, " for date: ", date_dataset, " channel: ", channel, " temperature: ", tempe)

                        # else:
                        #     print("Not enough voltage points for date: ", date_dataset, " channel: ", channel, " temperature: ", tempe)

        if output_path is not None:
            self.info_corrected_df.to_csv(output_path, sep=',', index=False, mode='w')
        else:
            print("Output path is not set, not saving the results.")
    
        # print(np.unique(self.info_corrected_df.date_str))
        self.info_corrected_Vbd = d2d.data(self.info_corrected_df)

        mask = (self.info_corrected_Vbd.breakdown_voltage_V > 0)
        # mask = ~np.isnan(self.info_corrected_Vbd.gain) & (self.info_corrected_Vbd.breakdown_voltage_V > 0)
        self.info_corrected_Vbd = self.info_corrected_Vbd.apply_mask(mask)
        # print("6: ", len(self.info_corrected_Vbd))
        # print(np.unique(self.info_corrected_Vbd.date_str))
        

### Load Data

In [ ]:
result_all = GainAnalysis(df_all, output_path = None)
result_all_df = result_all.info_corrected_Vbd.get_df()
info_corrected_Vbd = result_all.info_corrected_Vbd



### Finalized Plots

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(15, 8), sharex=True, height_ratios=[3, 1])
fig.subplots_adjust(hspace=0)

max_err = 0

for channel in np.unique(info_corrected_Vbd.channel):
    mask = info_corrected_Vbd.channel == channel
    tmp = info_corrected_Vbd.apply_mask(mask)
    # normalized_capacity = tmp.junction_capacity/tmp.junction_capacity.mean()
    ax[1].scatter(channel, -tmp.junction_capacity.std()/tmp.junction_capacity.mean()*100, color='black')

    max_err = max(max_err, -tmp.junction_capacity.std()/tmp.junction_capacity.mean()*100)

print(f"Maximum error: {max_err}")

for temperature in np.unique(info_corrected_Vbd.temperature_K):
    mask = info_corrected_Vbd.temperature_K == temperature
    temp_info = info_corrected_Vbd.apply_mask(mask)

    capacitance_list = []
    error_list = []
    temp_list = []


    for channel in np.unique(temp_info.channel):
        mask = temp_info.channel == channel
        tmp = temp_info.apply_mask(mask)

        temp_list.append(channel)
        capacitance_list.append(-tmp.junction_capacity.mean()*1e15)  # in fF
        error_list.append(tmp.junction_capacity.std()*1e15)  # in fF

    ax[0].errorbar(temp_list, (capacitance_list), yerr=error_list, fmt='o', 
                 label=f"{temperature} K", color = result_all.color_temperature[temperature])
    

xlabels = [f"{i}" for i in np.unique(info_corrected_Vbd.channel)]
ax[0].set_xticks(np.arange(len(xlabels)))
ax[0].set_xticklabels(xlabels)


ax[1].set_ylim(1, 5)

ax[1].set_xlabel("Channel")
ax[0].set_ylabel("$C_j$ [fF]")
ax[1].set_ylabel("Normalized $\sigma$ [%]")

mean = 101
error = 6
ax[0].axhline(mean, color='black', linestyle='--', linewidth=0.5)
ax[0].axhspan(mean-error, mean+error, color='gray', alpha=0.2, label="nEXO2022")

ax[0].legend(bbox_to_anchor=(0.5, 1.15), loc='upper center', ncol=6)



# plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/junction_capacity_vs_temperature_channel.pdf", dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(2,1, figsize=(15, 8), sharex=True, height_ratios=[3, 1])
fig.subplots_adjust(hspace=0)
max_err = 0

for channel in np.unique(info_corrected_Vbd.channel):
    mask = info_corrected_Vbd.channel == channel
    tmp = info_corrected_Vbd.apply_mask(mask)
    # normalized_capacity = tmp.junction_capacity/tmp.junction_capacity.mean()
    ax[1].scatter(channel, tmp.breakdown_voltage_V.std()/tmp.breakdown_voltage_V.mean()*100, color='black')

    max_err = max(max_err, tmp.breakdown_voltage_V.std()/tmp.breakdown_voltage_V.mean()*100)

print(f"Maximum error: {max_err}")

for temperature in np.unique(info_corrected_Vbd.temperature_K):
    mask = info_corrected_Vbd.temperature_K == temperature
    temp_info = info_corrected_Vbd.apply_mask(mask)

    capacitance_list = []
    error_list = []
    temp_list = []


    for channel in np.unique(temp_info.channel):
        mask = temp_info.channel == channel
        tmp = temp_info.apply_mask(mask)

        temp_list.append(channel)
        capacitance_list.append(-tmp.breakdown_voltage_V.mean())  # in fF
        error_list.append(tmp.breakdown_voltage_V.std())  # in fF

    ax[0].errorbar(temp_list, (capacitance_list), yerr=error_list, fmt='o', 
                 label=f"{temperature} K", color = result_all.color_temperature[temperature])
    

xlabels = [f"{i}" for i in np.unique(info_corrected_Vbd.channel)]
ax[0].set_xticks(np.arange(len(xlabels)))
ax[0].set_xticklabels(xlabels)


ax[1].set_ylim(0, 1.5)

ax[1].set_xlabel("Channel")
ax[0].set_ylabel("$V_{bd}$ [V]")
ax[1].set_ylabel("Normalized $\sigma$ [%]")

mean = -44.51
error = 0.05
ax[0].axhline(mean, color='black', linestyle='--', linewidth=0.5)
ax[0].axhspan(mean-error, mean+error, color='gray', alpha=0.2, label="nEXO2022")

ax[0].legend(bbox_to_anchor=(0.5, 1.15), loc='upper center', ncol=6)

# reverse the y-axis for breakdown voltage
ax[0].invert_yaxis()


# plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/breakdown_voltage_vs_temperature_channel.pdf", dpi=300, bbox_inches='tight')


In [ ]:
date_str_dict = {0:[datetime.datetime(2024, 5, 10),datetime.datetime(2024, 7, 1)],
                 1:[datetime.datetime(2024, 8, 8),datetime.datetime(2024, 9, 25)],
                 2:[datetime.datetime(2024, 10, 10),datetime.datetime(2024, 11, 5)]}

# convert to pd datetime64[ns]
date_str_dict = {k: [pd.to_datetime(v[0]), pd.to_datetime(v[1])] for k, v in date_str_dict.items()}

duration = np.array([])
for i in range(len(date_str_dict)):
    duration = np.append(duration, date_str_dict[i][1] - date_str_dict[i][0])

fig, ax = plt.subplots(2,3,figsize=(15, 8), sharex='col', sharey='row',width_ratios=duration/min(duration))
# gap between subplots
fig.subplots_adjust(wspace=0)
fig.subplots_adjust(hspace=0)


for i in range(len(date_str_dict)):
    mask = (info_corrected_Vbd.date_time >= date_str_dict[i][0]) & (info_corrected_Vbd.date_time <= date_str_dict[i][1])
    time_info = info_corrected_Vbd.apply_mask(mask)

    for channel in np.unique(time_info.channel):
        mask = time_info.channel == channel
        tmp = time_info.apply_mask(mask)
        
        ax[0][i].scatter(tmp.date_time, tmp.junction_capacity/tmp.junction_capacity.mean(), label=f"Channel {channel}", color = result_all.color_channel[channel])
        ax[1][i].scatter(tmp.date_time, tmp.breakdown_voltage_V/tmp.breakdown_voltage_V.mean(), label=f"Channel {channel}", color = result_all.color_channel[channel])

        # ax[0][i].scatter(tmp.date_time, tmp.junction_capacity, label=f"Channel {channel}", color = result_all.color_channel[channel])
        # ax[1][i].scatter(tmp.date_time, tmp.breakdown_voltage_V, label=f"Channel {channel}", color = result_all.color_channel[channel])
    
    # create xlabels with time stamps every 7 days
    xlabels = pd.date_range(start=date_str_dict[i][0]+pd.Timedelta(days=5), 
                            end=date_str_dict[i][1]-pd.Timedelta(days=5), freq='7D')
    
    ax[0][i].set_xlim(date_str_dict[i][0], date_str_dict[i][1])
    ax[1][i].set_xlim(date_str_dict[i][0], date_str_dict[i][1])
    # ax[0][i].set_ylim(0.97, 1.03)

    # rotate x-ticks for better visibility
    ax[1][i].set_xticks(xlabels)
    ax[1][i].set_xticklabels(xlabels, rotation=45, ha='right')

    # # set axis date format
    ax[1][i].xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y-%m-%d'))

ax[0][0].text(0.5, 0.85, "GXe-1", transform=ax[0][0].transAxes, ha='center')
ax[0][1].text(0.5, 0.85, "GXe-2", transform=ax[0][1].transAxes, ha='center')
ax[0][2].text(0.5, 0.85, "LXe-1", transform=ax[0][2].transAxes, ha='center')
    
fig.supxlabel('Date', y=-0.1)  # with adjusted position
ax[0][0].set_ylabel('Normalized $C_j$')  
ax[1][0].set_ylabel('Normalized \nbreakdown voltage')

plt.legend(bbox_to_anchor=(1, 2.5), ncol=6, fontsize=15)


# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/normaized_Cj_date_channel.pdf", dpi=300, bbox_inches='tight')
# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/variables_date_channel.png", dpi=300, bbox_inches='tight')


In [ ]:
date_str_dict = {0:[datetime.datetime(2024, 5, 10),datetime.datetime(2024, 7, 1)],
                 1:[datetime.datetime(2024, 8, 8),datetime.datetime(2024, 9, 25)],
                 2:[datetime.datetime(2024, 10, 10),datetime.datetime(2024, 11, 5)]}

# convert to pd datetime64[ns]
date_str_dict = {k: [pd.to_datetime(v[0]), pd.to_datetime(v[1])] for k, v in date_str_dict.items()}

duration = np.array([])
for i in range(len(date_str_dict)):
    duration = np.append(duration, date_str_dict[i][1] - date_str_dict[i][0])

fig, ax = plt.subplots(2,3,figsize=(15, 8), sharex='col', sharey='row',width_ratios=duration/min(duration))
# gap between subplots
fig.subplots_adjust(wspace=0)
fig.subplots_adjust(hspace=0)


for i in range(len(date_str_dict)):
    mask = (info_corrected_Vbd.date_time >= date_str_dict[i][0]) & (info_corrected_Vbd.date_time <= date_str_dict[i][1])
    time_info = info_corrected_Vbd.apply_mask(mask)

    for channel in np.unique(time_info.channel):
        for temperature in np.unique(time_info.temperature_K): 
            mask = (time_info.channel == channel) & (time_info.temperature_K == temperature)
            tmp = time_info.apply_mask(mask)

            ax[0][i].scatter(tmp.date_time, tmp.junction_capacity/tmp.junction_capacity.mean(), label=f"Channel {channel}", color = result_all.color_channel[channel])
            ax[1][i].scatter(tmp.date_time, tmp.breakdown_voltage_V/tmp.breakdown_voltage_V.mean(), label=f"Channel {channel}", color = result_all.color_channel[channel])

            # ax[0][i].scatter(tmp.date_time, tmp.junction_capacity, label=f"Channel {channel}", color = result_all.color_channel[channel])
            # ax[1][i].scatter(tmp.date_time, tmp.breakdown_voltage_V, label=f"Channel {channel}", color = result_all.color_channel[channel])
    
    # create xlabels with time stamps every 7 days
    xlabels = pd.date_range(start=date_str_dict[i][0]+pd.Timedelta(days=5), 
                            end=date_str_dict[i][1]-pd.Timedelta(days=5), freq='7D')
    
    ax[0][i].set_xlim(date_str_dict[i][0], date_str_dict[i][1])
    ax[1][i].set_xlim(date_str_dict[i][0], date_str_dict[i][1])
    # ax[0][i].set_ylim(0.97, 1.03)

    # rotate x-ticks for better visibility
    ax[1][i].set_xticks(xlabels)
    ax[1][i].set_xticklabels(xlabels, rotation=45, ha='right')

    # # set axis date format
    ax[1][i].xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y-%m-%d'))

ax[0][0].text(0.5, 0.85, "GXe-1", transform=ax[0][0].transAxes, ha='center')
ax[0][1].text(0.5, 0.85, "GXe-2", transform=ax[0][1].transAxes, ha='center')
ax[0][2].text(0.5, 0.85, "LXe-1", transform=ax[0][2].transAxes, ha='center')
    
fig.supxlabel('Date', y=-0.1)  # with adjusted position
ax[0][0].set_ylabel('Normalized $C_j$')  
ax[1][0].set_ylabel('Normalized \nbreakdown voltage')

plt.legend(bbox_to_anchor=(1, 2.5), ncol=6, fontsize=15)


# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/normaized_Cj_date_channel.pdf", dpi=300, bbox_inches='tight')
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/variables_date_channel.png", dpi=300, bbox_inches='tight')


In [ ]:
### Each channel

fig, ax = plt.subplots(figsize=(15, 7))

percentage_err = []

for voltage in np.unique(info_corrected_Vbd.voltage_preamp1_V):
   
    # mask = ~np.isnan(info.spe_position)
    mask = ~np.isnan(info_corrected_Vbd.spe_position) & (info_corrected_Vbd.voltage_preamp1_V == voltage)
    # mask = info.temperature_K == temperature
    SPE_info = info_corrected_Vbd.copy()
    SPE_info.apply_mask(mask, inplace=True, dry=False)

    spe_position_list = []
    spe_position_err_list = []
    channel_list = np.unique(SPE_info.channel)
    for channel in channel_list:
        mask = SPE_info.channel == channel
        tmp = SPE_info.apply_mask(mask)
        
        spe_position_avg = tmp.spe_position.mean()
        spe_position_avg_err = np.std(tmp.spe_position)

        if (voltage == -46) & ((channel == 0) | (channel == 1) | (channel == 11)):
            spe_position_list.append(spe_position_avg/2)
            spe_position_err_list.append(spe_position_avg_err/2)
        else: 
            spe_position_list.append(spe_position_avg)
            spe_position_err_list.append(spe_position_avg_err)


    diff = max(spe_position_list) - min(spe_position_list)
    perc_diff = diff / min(spe_position_list) * 100
    print(f"Voltage: {voltage:.1f} V, difference in SPE position: {diff:.2f} [V]. Percentage difference: {perc_diff:.2f} %")
    
    ax.errorbar(channel_list, spe_position_list, 
                    yerr=spe_position_err_list, 
                    fmt="o", 
                    label=f"{voltage:.1f} V",
                    markersize=8,

                    )

    percentage_err.append(np.array(spe_position_err_list)/np.array(spe_position_list)*100)

# plt.legend(bbox_to_anchor = (1,1.1))
# plt.gca().invert_xaxis()
plt.ylabel("SPE position [V s]")
plt.xlabel("SiPM channel")
plt.legend(bbox_to_anchor = (1,1), loc="upper left", title="Bias voltage")
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/spe_position_v2.pdf", dpi=300, bbox_inches='tight')


In [ ]:
result_LXe = GainAnalysis(df_LXe, output_path = None)
result_LXe_df = result_LXe.info_corrected_Vbd.get_df()
info_corrected_Vbd_LXe = result_LXe.info_corrected_Vbd

### Each channel

fig, ax = plt.subplots(figsize=(15, 7))

for voltage in np.unique(info_corrected_Vbd_LXe.voltage_preamp1_V):
   
    # mask = ~np.isnan(info.spe_position)
    mask = ~np.isnan(info_corrected_Vbd_LXe.spe_position) & (info_corrected_Vbd_LXe.voltage_preamp1_V == voltage)
    # mask = info.temperature_K == temperature
    SPE_info = info_corrected_Vbd_LXe.copy()
    SPE_info.apply_mask(mask, inplace=True, dry=False)

    spe_position_list = []
    spe_position_err_list = []
    channel_list = np.unique(SPE_info.channel)
    for channel in channel_list:
        mask = SPE_info.channel == channel
        tmp = SPE_info.apply_mask(mask)
        
        spe_position_avg = tmp.spe_position.mean()
        spe_position_avg_err = np.std(tmp.spe_position)

        if (voltage == -46) & ((channel == 0) | (channel == 1) | (channel == 11)):
            spe_position_list.append(spe_position_avg/2)
            spe_position_err_list.append(spe_position_avg_err/2)
        else: 
            spe_position_list.append(spe_position_avg)
            spe_position_err_list.append(spe_position_avg_err)

    diff = max(spe_position_list) - min(spe_position_list)
    perc_diff = diff / min(spe_position_list) * 100
    print(f"Voltage: {voltage:.1f} V, difference in SPE position: {diff:.2f} [V]. Percentage difference: {perc_diff:.2f} %")
    
    ax.errorbar(channel_list, spe_position_list, 
                    yerr=spe_position_err_list, 
                    fmt="o", 
                    label=f"{voltage:.1f} V",
                    markersize=8,

                    )

# plt.legend(bbox_to_anchor = (1,1.1))
# plt.gca().invert_xaxis()
plt.ylabel("SPE position [V s]")
plt.xlabel("SiPM channel")
plt.legend(bbox_to_anchor = (1,1), loc="upper left", title="Bias voltage")
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/spe_position_v2.pdf", dpi=300, bbox_inches='tight')



In [ ]:
# Save all SPE position to csv
result_LXe = GainAnalysis(df_LXe, output_path = None)
result_LXe_df = result_LXe.info_corrected_Vbd.get_df()
info_corrected_Vbd_LXe = result_LXe.info_corrected_Vbd

df = pd.DataFrame(columns=["voltage_preamp1_V", "channel", "spe_position", "spe_position_err"])

for voltage in np.unique(info_corrected_Vbd_LXe.voltage_preamp1_V):
   
    # mask = ~np.isnan(info.spe_position)
    mask = ~np.isnan(info_corrected_Vbd_LXe.spe_position) & (info_corrected_Vbd_LXe.voltage_preamp1_V == voltage)
    # mask = info.temperature_K == temperature
    SPE_info = info_corrected_Vbd_LXe.copy()
    SPE_info.apply_mask(mask, inplace=True, dry=False)

    spe_position_list = []
    spe_position_err_list = []
    channel_list = np.unique(SPE_info.channel)
    for channel in channel_list:
        mask = SPE_info.channel == channel
        tmp = SPE_info.apply_mask(mask)
        
        spe_position_avg = tmp.spe_position.mean()
        spe_position_avg_err = np.std(tmp.spe_position)

        if (voltage == -46) & ((channel == 0) | (channel == 1) | (channel == 11)):
            spe_position_list.append(spe_position_avg/2)
            spe_position_err_list.append(spe_position_avg_err/2)
        else: 
            spe_position_list.append(spe_position_avg)
            spe_position_err_list.append(spe_position_avg_err)

        df = df.append({"voltage_preamp1_V": voltage, "channel": int(channel), "spe_position": spe_position_avg, "spe_position_err": spe_position_avg_err}, ignore_index=True)

# df.to_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spe_position_LXe_2.csv", 
#           index=False, header=True)


#### Example V_bd plot

In [ ]:
mask = info_corrected_Vbd.breakdown_voltage_V == np.unique(info_corrected_Vbd.breakdown_voltage_V)[50]
tmp = info_corrected_Vbd.apply_mask(mask)

fig, ax = plt.subplots(figsize=(15,4))

assert len(np.unique(tmp.breakdown_voltage_V)) == 1
assert len(np.unique(tmp.total_capacity)) == 1

coeff_0 = tmp.total_capacity[0]/1.6e-19
coeff_1 = tmp.breakdown_voltage_V[0]*coeff_0

# draw the fit line
x = np.linspace(tmp.voltage_preamp1_V.min(), -43, 100)
y = coeff_0 * x + coeff_1

ax.scatter(tmp.voltage_preamp1_V, tmp.gain, s=20, color="#377eb8", label="data")
ax.plot(x, y, '--', label=f"fit: $M ={coeff_0:.2e} \cdot V_{{bias}}{coeff_1:.2e}$")
# ax.axhline(0, color='black')

ax.set_ylim(0, 1.8e8)

ax.axvline(-tmp.breakdown_voltage_V[0], linestyle='-.', 
           label=f"$V_{{bd}} = {tmp.breakdown_voltage_V[0]:.2f} V$")


# reverse x-axis
plt.gca().invert_xaxis()
plt.xlabel("Bias voltage [V]")
plt.ylabel("Gain")

# fig.set_figwidth(10)

plt.legend(bbox_to_anchor=(0.4, 0.5), loc='lower center')

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_fit.pdf", dpi=300, bbox_inches='tight')

#### Comparison with Literature

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))

df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/nEXO_2022_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

color_list = ["#ff7f00", "#984ea3"]

for i, col_name in enumerate(voltage_col_name):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = 1e3 * df_lit[gain_col_name[i]]
    ax.plot(bias_voltage, 
            gain,
            "o-",
            label = f"nEXO2022: {temperature_label}",
                    zorder=10,
                    color=color_list[i]
            )
    
    
df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/baudis_2023_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

for i, col_name in enumerate(voltage_col_name[0:1]):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = df_lit[gain_col_name[i]] # baudis 2018
    ax.plot(bias_voltage, 
            gain,
            "o-",
            markersize = 10,
            label = f"Peres2023: {temperature_label}",
                zorder=10,
                alpha = 0.8,
                markeredgecolor="black",
                color = "yellow"
            )


temperature = np.unique(info_corrected_Vbd.temperature_K)

mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
gain_info = info_corrected_Vbd.apply_mask(mask)


for i, channel in enumerate(np.unique(gain_info.channel)):
    
    mask = (gain_info.channel == channel)
    channel_info = gain_info.apply_mask(mask)
    
    for tempe in temperature:

        # print(temperature)
        mask = channel_info.temperature_K == tempe
        masked_temp = channel_info.apply_mask(mask)
        
        gain_list = []
        over_voltage_list = []
        run_id_list = []
        
        for j in range(len(masked_temp)):
            
            #gain = masked_temp.gain.mean()
            gain_list.append(masked_temp.gain/33)
            
            over_voltage_list.append(masked_temp.over_voltage_V)

        over_voltage = np.array(over_voltage_list).mean()
        gain = np.array(gain_list).mean()

        # if i == 0:
        plt.plot(over_voltage_list, gain_list, 
                "o-", 
                color=result_all.color_temperature[tempe],
                markersize=1
                )
            
        # else:
        #     plt.plot(over_voltage_list, gain_list, 
        #             "o-", 
        #             color=color_temperature[temperature],
        #             
        # plt.errorbar(tmp.voltage_preamp1_V, tmp.gain, 
        #          yerr=tmp.gain_err, 
        #          label = f"{temperature} K", 
        #          fmt="o", 
        #          ecolor = "black", 
        #          capsize=3,
        #          color=result_all.color_temperature[tempe]
        #          )


# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"this work: {temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=10)

plt.legend(bbox_to_anchor = (1,1.1), ncol=1)
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("Gain")
plt.xlabel("Over Voltage [V]")

# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_over_voltage_v2.pdf", dpi=100, bbox_inches='tight')
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_over_voltage_v2.png")


### Trashed Plots

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

for channel in np.unique(info_corrected_Vbd.channel):
    mask = info_corrected_Vbd.channel == channel
    tmp = info_corrected_Vbd.apply_mask(mask)
    normalized_capacity = tmp.junction_capacity/tmp.junction_capacity.mean()
    ax.scatter(tmp.date_time, normalized_capacity, label=f"Channel {channel}", color = result_all.color_channel[channel])

    print("Channel", channel, " Junction capacity spread: ", abs(normalized_capacity).std())

ax.set_ylim(0.8,1.2)

plt.xlabel("Date")
plt.ylabel("Normalized $C_j$ [fF]")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2)
plt.grid()


# plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/normaized_Cj_date_channel.pdf", dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))


capacitance_list = []
error_list = []
temp_list = []

for temperature in np.unique(info_corrected_Vbd.temperature_K):
    mask = info_corrected_Vbd.temperature_K == temperature
    tmp = info_corrected_Vbd.apply_mask(mask)

    temp_list.append(temperature)
    capacitance_list.append(-tmp.total_capacity.mean()*1e15)  # in fF
    error_list.append(tmp.total_capacity.std()*1e15)  # in fF

plt.errorbar(temp_list, (capacitance_list), yerr=error_list, fmt='o')

plt.xlabel("Temperature [K]")
plt.ylabel("$C_j\,$[fF]")
plt.legend(bbox_to_anchor=(1., 1), loc='upper left', ncol=1)
plt.grid()
# plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/junction_capacity_vs_date_temperature.pdf", dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))


capacitance_list = []
error_list = []
temp_list = []

for temperature in np.unique(info_corrected_Vbd.temperature_K):
    mask = info_corrected_Vbd.temperature_K == temperature
    tmp = info_corrected_Vbd.apply_mask(mask)

    temp_list.append(temperature)
    capacitance_list.append(tmp.breakdown_voltage_V.mean())  # in fF
    error_list.append(tmp.breakdown_voltage_V.std())  # in fF

plt.errorbar(temp_list, capacitance_list, yerr=error_list, fmt='o')

plt.xlabel("Temperature [K]")
plt.ylabel("$C_j\,$[fF]")
plt.legend(bbox_to_anchor=(1., 1), loc='upper left', ncol=1)
plt.grid()
# plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')
# # # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/junction_capacity_vs_date_temperature.pdf", dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))


capacitance_list = []
error_list = []
temp_list = []

for temperature in np.unique(info_corrected_Vbd.temperature_K):
    mask = info_corrected_Vbd.temperature_K == temperature
    tmp = info_corrected_Vbd.apply_mask(mask)

    temp_list.append(temperature)
    capacitance_list.append(-tmp.total_capacity.mean()*1e15/33)  # in fF
    error_list.append(tmp.total_capacity.std()*1e15/33)  # in fF

plt.errorbar(temp_list, (capacitance_list), yerr=error_list, fmt='o')

plt.xlabel("Temperature [K]")
plt.ylabel("$C_j\,$[fF]")
plt.legend(bbox_to_anchor=(1., 1), loc='upper left', ncol=1)
plt.grid()
# plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')
# # # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/junction_capacity_vs_date_temperature.pdf", dpi=300, bbox_inches='tight')


In [ ]:


for temperature in np.unique(info_corrected_Vbd.temperature_K):
    mask = info_corrected_Vbd.temperature_K == temperature
    temp_info = info_corrected_Vbd.apply_mask(mask)

    capacitance_list = []
    error_list = []
    temp_list = []


    for channel in np.unique(temp_info.channel):
        mask = temp_info.channel == channel
        tmp = temp_info.apply_mask(mask)

        temp_list.append(channel)
        capacitance_list.append(-tmp.junction_capacity.mean()*1e15)  # in fF
        error_list.append(tmp.junction_capacity.std()*1e15)  # in fF

    print("Temperature: ", temperature, " Channels: ", np.unique(temp_info.channel))

    plt.errorbar(temp_list, (capacitance_list), yerr=error_list, fmt='o', 
                 label=f"{temperature} K", color = result_all.color_temperature[temperature])

plt.xlabel("Temperature [K]")
plt.ylabel("$C_j\,$[fF]")
plt.legend(bbox_to_anchor=(0, 1.15), loc='upper left', ncol=5)
plt.grid()
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/junction_capacity_vs_temperature_channel.pdf", dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))


capacitance_list = []
error_list = []
temp_list = []

for temperature in np.unique(info_corrected_Vbd.temperature_K):
    mask = info_corrected_Vbd.temperature_K == temperature
    tmp = info_corrected_Vbd.apply_mask(mask)

    temp_list.append(temperature)
    capacitance_list.append(-tmp.junction_capacity.mean()*1e15)  # in fF
    error_list.append(tmp.junction_capacity.std()*1e15)  # in fF

plt.errorbar(temp_list, (capacitance_list), yerr=error_list, fmt='o')

plt.xlabel("Temperature [K]")
plt.ylabel("$C_j\,$[fF]")
plt.legend(bbox_to_anchor=(1., 1), loc='upper left', ncol=1)
plt.grid()
# plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')
# # # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/junction_capacity_vs_date_temperature.pdf", dpi=300, bbox_inches='tight')


In [ ]:
for temperature in np.unique(info_corrected_Vbd.temperature_K):
    mask = info_corrected_Vbd.temperature_K == temperature
    tmp = info_corrected_Vbd.apply_mask(mask)
    # plt.plot(tmp.junction_capacity/tmp.junction_capacity[0], label=f"Channel {channel}", color = result_all.color_channel[channel])
    # plt.scatter(tmp.date_time, (tmp.junction_capacity-tmp.junction_capacity[0])/tmp.junction_capacity[0], label=f"Channel {channel}", color = result_all.color_channel[channel])
    # plt.scatter(tmp.date_time, (tmp.junction_capacity)/tmp.junction_capacity[0], label=f"Temperature {temperature} K", color = result_all.color_temperature[temperature])
    plt.scatter(tmp.date_time, (tmp.junction_capacity)*1e12, label=f"{temperature} K", color = result_all.color_temperature[temperature])

plt.xlabel("Date")
plt.ylabel("$C_j$ [pF]")
plt.legend(bbox_to_anchor=(1., 1), loc='upper left', ncol=1)
plt.grid()
# plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/junction_capacity_vs_date_temperature.pdf", dpi=300, bbox_inches='tight')


In [ ]:
for channel in np.unique(info_corrected_Vbd.channel):
    mask = info_corrected_Vbd.channel == channel
    tmp = info_corrected_Vbd.apply_mask(mask)
    # plt.plot(tmp.junction_capacity/tmp.junction_capacity[0], label=f"Channel {channel}", color = result_all.color_channel[channel])
    # plt.scatter(tmp.date_time, (tmp.junction_capacity-tmp.junction_capacity[0])/tmp.junction_capacity[0], label=f"Channel {channel}", color = result_all.color_channel[channel])
    plt.scatter(tmp.date_time, (tmp.junction_capacity)/tmp.junction_capacity[0], label=f"Channel {channel}", color = result_all.color_channel[channel])

plt.xlabel("Date")
plt.ylabel("Normalized $C_j$")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2)
plt.grid()
# plt.show()
# plt.scatter(info_corrected_Vbd.channel, info_corrected_Vbd.junction_capacity, color='red')
# # # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/junction_capacity_vs_date_channel.pdf", dpi=300, bbox_inches='tight')


In [ ]:
mask = info_corrected_Vbd.temperature_K == 175
tmp = info_corrected_Vbd.apply_mask(mask)

plt.scatter(tmp.date_time, tmp.breakdown_voltage_V)

In [ ]:
plt.scatter(info_corrected_Vbd.temperature_K, info_corrected_Vbd.breakdown_voltage_V)

In [ ]:
plt.scatter(info_corrected_Vbd.date_time, info_corrected_Vbd.breakdown_voltage_V)

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))

df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/nEXO_2022_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

for i, col_name in enumerate(voltage_col_name):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = 1e3 * df_lit[gain_col_name[i]]
    ax.plot(bias_voltage, 
            gain,
            "o-",
            label = f"nEXO2022: {temperature_label}",
                    zorder=10
            )
    
    
df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/baudis_2018_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

for i, col_name in enumerate(voltage_col_name):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = 1e6 * df_lit[gain_col_name[i]] # baudis 2018
    ax.plot(bias_voltage, 
            gain,
            "o-",
            label = f"baudis2018: {temperature_label}",
                    zorder=10
            )
    
df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/baudis_2023_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

for i, col_name in enumerate(voltage_col_name):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = df_lit[gain_col_name[i]] # baudis 2018
    ax.plot(bias_voltage, 
            gain,
            "o-",
            label = f"baudis2023: {temperature_label}",
                    zorder=10
            )


temperature = np.unique(info_corrected_Vbd.temperature_K)

mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
gain_info = info_corrected_Vbd.apply_mask(mask)


for tempe in temperature:
    
    mask = gain_info.temperature_K == tempe
    masked_temp = gain_info.apply_mask(mask)
    coef0_list = []
    coef1_list = []
    
    for i, channel in enumerate(np.unique(masked_temp.channel)):
        gain_list = []
        over_voltage_list = []
        run_id_list = []
        
        
        mask = (masked_temp.channel == channel)
        masked_info = masked_temp.apply_mask(mask)
                
        for j in range(len(masked_info)):
            # gain = masked_info.gain.mean()
            gain_list.append(masked_info.gain/33)
            over_voltage_list.append(masked_info.over_voltage_V)
            coef0_list.append(masked_info.coef0[0])
            coef1_list.append(masked_info.coef1[0])

        over_voltage = np.array(over_voltage_list).mean()
        gain = np.array(gain_list).mean()
        
        ax.plot(over_voltage_list, gain_list, 
                "o-", 
                color=result_all.color_temperature[tempe]
                )
        
    m = np.array(coef0_list).mean()
    c = np.array(coef1_list).mean()
    print("Temperature: ", tempe, " Coefficients: ", m, c)
    x = np.arange(0, 7, 3)
    y = -m/33 * x
    ax.plot(x, y, '-', label=f"{tempe} K")
        

# add items to legend
for temp in temperature:
    ax.scatter([], [], label=f"NUXE-3: {temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=50)

plt.legend(bbox_to_anchor = (1,1.1), ncol=2, loc="upper left")
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("Gain")
plt.xlabel("Over Voltage [V]")
# # # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_over_voltage.pdf", dpi=300, bbox_inches='tight')


In [ ]:
fig,ax = plt.subplots(figsize=(10,6))

mask = ~np.isnan(info_corrected_Vbd.gain)
gain_info = info_corrected_Vbd.apply_mask(mask)

# temperatures
temperature = np.unique(info_corrected_Vbd.temperature_K)

# select data before and after change of power supply
mask = gain_info.date_time < result_all.date_power_supply_changed
masked_before = gain_info.apply_mask(mask)

mask = gain_info.date_time > result_all.date_power_supply_changed
masked_after = gain_info.apply_mask(mask)

selected_data = [masked_before, masked_after]


for data_set in selected_data:
    for tempe in temperature:
        mask = data_set.temperature_K == tempe
        masked_temp = data_set.apply_mask(mask)

        for i, channel in enumerate(np.unique(masked_temp.channel)):
            mask = masked_temp.channel == channel
            masked_channel = masked_temp.apply_mask(mask)
            
            for date_dataset in np.sort(np.unique(masked_channel.date_str)):
                mask = masked_channel.date_str == date_dataset
                masked_date = masked_channel.apply_mask(mask)

                voltage_list = np.unique(masked_date.voltage_preamp1_V)
                # n_points.append(len(voltage_list))
                # date_str_list.append(date_dataset)
                
                if len(voltage_list) >= 3:
                    
                    # fit
                    coef, res, _, _, _ = np.polyfit(masked_date.voltage_preamp1_V,masked_date.gain,1, full=True)
                    poly1d_fn = np.poly1d(coef)

                    breakdown_voltage = coef[1]/coef[0]
                    
                    voltage_list = np.arange(-52,-42,2)
                    y_output = voltage_list*coef[0] + coef[1]
                    # ax.plot(-voltage_list, poly1d_fn(voltage_list), '--',
                    ax.plot(voltage_list,y_output, '--',
                            color=result_all.color_channel[channel]) 
                    ax.plot(-breakdown_voltage, 0, 'o',
                            color=result_all.color_channel[channel]) 
                    
    
# info_corrected_Vbd = d2d.data(info_corrected_df)

# print(len(info_corrected_Vbd))

plt.legend(bbox_to_anchor = (1,1.1),title="Channels", ncol=2)
# plt.gca().invert_xaxis()
plt.title(f"Temperature at {temperature} K")
plt.ylabel("Gain")
plt.xlabel("Bias Voltage")

In [ ]:
temperature = np.unique(info_corrected_Vbd.temperature_K)

for tempe in temperature:

    mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
    gain_info = info_corrected_Vbd.apply_mask(mask)

    # print(temperature)
    mask = gain_info.temperature_K == tempe
    masked_temp = gain_info.apply_mask(mask)
    
    for i, channel in enumerate(np.unique(masked_temp.channel)):
        gain_list = []
        over_voltage_list = []
        run_id_list = []
        
        mask = (masked_temp.channel == channel)
        masked_info = masked_temp.apply_mask(mask)
                
        # for j in range(len(masked_info)):
        #     # gain = masked_info.gain.mean()
        #     gain_list.append(masked_info.gain/33)
            
        #     over_voltage_list.append(masked_info.over_voltage_V)
        
        # plt.plot(over_voltage_list, gain_list, 
        # plt.plot(masked_info.over_voltage_V, masked_info.gain/33,
        plt.plot(masked_info.date_time, masked_info.breakdown_voltage_V,
                "o", 
                color=result_all.color_temperature[tempe]
                )

# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"{temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=50)

plt.legend(bbox_to_anchor = (1,1.1))
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("Breakdown Voltage [V]")
plt.xlabel("Date")
plt.xticks(rotation=45, ha='right')
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/breakdown_voltage_vs_date_temperature.pdf", dpi=300, bbox_inches='tight')


In [ ]:
temperature = np.unique(info_corrected_Vbd.temperature_K)

for tempe in temperature:

    mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
    gain_info = info_corrected_Vbd.apply_mask(mask)

    # print(temperature)
    mask = gain_info.temperature_K == tempe
    masked_temp = gain_info.apply_mask(mask)
    
    for i, channel in enumerate(np.unique(masked_temp.channel)):
        gain_list = []
        over_voltage_list = []
        run_id_list = []
        
        mask = (masked_temp.channel == channel)
        masked_info = masked_temp.apply_mask(mask)
                
        # for j in range(len(masked_info)):
        #     # gain = masked_info.gain.mean()
        #     gain_list.append(masked_info.gain/33)
            
        #     over_voltage_list.append(masked_info.over_voltage_V)
        
        # plt.plot(over_voltage_list, gain_list, 
        # plt.plot(masked_info.over_voltage_V, masked_info.gain/33,
        plt.plot(masked_info.date_time, masked_info.junction_capacity,
                "o", 
                color=result_all.color_temperature[tempe]
                )

# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"{temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=50)

plt.legend(bbox_to_anchor = (1,1.1))
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("$C_j$ [F]")
plt.xlabel("Date")
plt.xticks(rotation=45, ha='right')
# # plt.savefig("junction_capacity_vs_date_temperature.pdf", dpi=300, bbox_inches='tight')
